In [2]:
# ================= 环境准备 =================
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langgraph.prebuilt import create_react_agent
from langchain.tools import tool
from datetime import datetime
from pydantic import BaseModel, Field, field_validator   # 注意导入 field_validator

load_dotenv()

# ================= 定义 Pydantic 输入模型（V2 语法） =================
class DateDiffInput(BaseModel):
    start_date: str = Field(description="起始日期，格式 YYYY-MM-DD，例如 2025-05-01")
    end_date: str = Field(default=None, description="结束日期，格式 YYYY-MM-DD，默认当前日期")

    # V2 中使用 @field_validator，注意写法
    @field_validator('start_date', 'end_date', mode='before')
    @classmethod
    def validate_date(cls, v):
        if v is None:
            return v
        try:
            datetime.strptime(v, "%Y-%m-%d")
        except ValueError:
            raise ValueError(f"日期 '{v}' 格式错误，请使用 YYYY-MM-DD")
        return v

# ================= 工具1：获取当前时间 =================
@tool
def get_current_time(format: str = "YYYY-MM-DD HH:MM:SS") -> str:
    """
    获取当前的日期和时间。
    
    Args:
        format: 时间格式，支持两种：
                "YYYY-MM-DD HH:MM:SS" -> 2025-05-21 14:30:00
                "timestamp" -> 1747845000
    """
    try:
        now = datetime.now()
        if format == "timestamp":
            return str(int(now.timestamp()))
        else:
            return now.strftime("%Y-%m-%d %H:%M:%S")
    except Exception as e:
        return f"获取时间失败：{str(e)}"

# ================= 工具2：计算日期差（使用 args_schema 强校验） =================
@tool(args_schema=DateDiffInput)
def calculate_days_between(start_date: str, end_date: str = None) -> str:
    """
    计算两个日期之间的天数差。
    """
    try:
        start = datetime.strptime(start_date, "%Y-%m-%d")
        if end_date:
            end = datetime.strptime(end_date, "%Y-%m-%d")
        else:
            end = datetime.now()
        delta = abs((end - start).days)
        return f"从 {start_date} 到 {end_date if end_date else '今天'} 相差 {delta} 天。"
    except Exception as e:
        return f"计算失败：{str(e)}"

# ================= 创建 Agent =================
llm = ChatOpenAI(
    model="deepseek-chat",
    api_key=os.getenv("DEEPSEEK_API_KEY"),
    base_url="https://api.deepseek.com/v1",
    temperature=0
)

tools = [get_current_time, calculate_days_between]
agent_executor = create_react_agent(llm, tools)



C:\Users\limin\AppData\Local\Temp\ipykernel_10964\2493572975.py:75: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent_executor = create_react_agent(llm, tools)


In [3]:
# ================= 测试正常任务 =================
print("=" * 60)
print("测试1：正常输入 - 使用正确日期格式")
user_input = "从 2025-05-01 到今天过了多少天？"
print(f"用户: {user_input}\n")
for event in agent_executor.stream(
    {"messages": [("user", user_input)]},
    {"recursion_limit": 15}
):
    for value in event.values():
        value["messages"][-1].pretty_print()



测试1：正常输入 - 使用正确日期格式
用户: 从 2025-05-01 到今天过了多少天？

================================== Ai Message ==================================

好的，我先获取当前时间。
Tool Calls:
  get_current_time (call_00_bkbvCNx3PguE47O4nnGl6426)
 Call ID: call_00_bkbvCNx3PguE47O4nnGl6426
  Args:
    format: YYYY-MM-DD HH:MM:SS
================================= Tool Message =================================
Name: get_current_time

2026-05-21 18:31:37
================================== Ai Message ==================================

当前日期是 **2026年5月21日**。接下来计算从 2025-05-01 到 2026-05-21 的天数差。
Tool Calls:
  calculate_days_between (call_00_WauYvaJRB3FWoPOwu9Ev0191)
 Call ID: call_00_WauYvaJRB3FWoPOwu9Ev0191
  Args:
    start_date: 2025-05-01
    end_date: 2026-05-21
================================= Tool Message =================================
Name: calculate_days_between

从 2025-05-01 到 2026-05-21 相差 385 天。
================================== Ai Message ==================================

从 **2025年5月1日** 到 **今天（2026年5月21日）**，一共

In [5]:
# ================= 测试异常输入 =================
print("\n" + "=" * 60)
print("测试2：异常输入 - 错误的日期格式")
user_input = "计算从 2025/05/01 到今天的相差天数"
print(f"用户: {user_input}\n")
for event in agent_executor.stream(
    {"messages": [("user", user_input)]},
    {"recursion_limit": 15}
):
    for value in event.values():
        value["messages"][-1].pretty_print()




测试2：异常输入 - 错误的日期格式
用户: 计算从 2025/05/01 到今天的相差天数

================================== Ai Message ==================================

好的，我先获取今天的日期。
Tool Calls:
  get_current_time (call_00_QX33241zctkMirD3pd0s8528)
 Call ID: call_00_QX33241zctkMirD3pd0s8528
  Args:
    format: YYYY-MM-DD HH:MM:SS
================================= Tool Message =================================
Name: get_current_time

2026-05-21 18:32:44
================================== Ai Message ==================================

今天是 **2026年5月21日**。现在来计算从 2025年5月1日 到今天的相差天数。
Tool Calls:
  calculate_days_between (call_00_Y8aCSgtWp2IAdI3oU9gS7697)
 Call ID: call_00_Y8aCSgtWp2IAdI3oU9gS7697
  Args:
    start_date: 2025-05-01
    end_date: 2026-05-21
================================= Tool Message =================================
Name: calculate_days_between

从 2025-05-01 到 2026-05-21 相差 385 天。
================================== Ai Message ==================================

从 **2025年5月1日** 到 **2026年5月21日**，一共相差 **385 天**。


In [6]:
# ================= 测试复杂任务（需要两个工具协作） =================
print("\n" + "=" * 60)
print("测试3：组合任务 - 先获取当前时间，再计算天数差")
user_input = "现在是什么时间？然后帮我算一下从 2025-05-01 到今天过了多少天？"
print(f"用户: {user_input}\n")
for event in agent_executor.stream(
    {"messages": [("user", user_input)]},
    {"recursion_limit": 15}
):
    for value in event.values():
        value["messages"][-1].pretty_print()


测试3：组合任务 - 先获取当前时间，再计算天数差
用户: 现在是什么时间？然后帮我算一下从 2025-05-01 到今天过了多少天？

================================== Ai Message ==================================

好的，我先获取当前时间。
Tool Calls:
  get_current_time (call_00_Jcmj3PZ6dAjkHm6jGr4B0587)
 Call ID: call_00_Jcmj3PZ6dAjkHm6jGr4B0587
  Args:
    format: YYYY-MM-DD HH:MM:SS
================================= Tool Message =================================
Name: get_current_time

2026-05-21 18:34:13
================================== Ai Message ==================================

当前时间是 **2026年5月21日 18:34:13**。

接下来我计算从 2025-05-01 到今天的天数差：
Tool Calls:
  calculate_days_between (call_00_bKjslNfTs7whsXaPvMb01351)
 Call ID: call_00_bKjslNfTs7whsXaPvMb01351
  Args:
    start_date: 2025-05-01
    end_date: 2026-05-21
================================= Tool Message =================================
Name: calculate_days_between

从 2025-05-01 到 2026-05-21 相差 385 天。
================================== Ai Message ==================================

### 结果汇总

| 项目 